# File Search

*Notebook 09*

Give your agent access to your own documents using OpenAI's built-in vector search.


---

## 🔧 Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from openai import OpenAI
from agents import Agent, FileSearchTool, Runner, ToolCallItem

MODEL = "gpt-5-mini"

# OpenAI client for vector store management
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def used_file_search(result) -> bool:
    """True if the run actually invoked the hosted file search tool."""
    return any(
        isinstance(item, ToolCallItem)
        and getattr(item.raw_item, "type", None) == "file_search_call"
        for item in result.new_items
    )


print("✅ Ready!")

---

## 🎯 The Problem

Web search finds public information, but not your own documents.

File Search gives your agent access to your private files.

It uses semantic and keyword search to retrieve relevant chunks as grounded context.

---

## 📚 Part 1: What Is a Vector Store?

File Search chunks documents and indexes them for semantic and keyword search.

At query time, it adds relevant chunks to context as evidence.

This is **RAG** (Retrieval-Augmented Generation): retrieve document evidence, then generate an answer from it.

Vectors represent text numerically, so similar passages can rank close even when the words differ.

Lesson 19 builds vector memory with ChromaDB.

### 💡 Cost Note

File Search storage and tool calls are billed at current rates: see the [pricing page](https://developers.openai.com/api/docs/pricing#built-in-tools).

Uploaded files and vector stores persist until deleted.

Run the cleanup cell at the end of the notebook when you're done.

---

## 📄 Part 2: Create a Sample Document

We'll create a sample product FAQ document to use as our knowledge base.

In [ ]:
# Create a sample document to upload
sample_doc = """# TechGadget Pro: Product FAQ

## What is TechGadget Pro?
TechGadget Pro is a smart productivity device that combines a noise-cancelling speaker,
AI assistant, and wireless charger in one compact unit. It connects via Bluetooth 5.3
and has a 12-hour battery life.

## What devices are compatible?
TechGadget Pro is compatible with iOS 15+, Android 11+, Windows 10+, and macOS 12+.
It supports up to 3 simultaneous Bluetooth connections.

## How do I reset TechGadget Pro?
To perform a factory reset: hold the power button and volume down button simultaneously
for 10 seconds until the LED flashes red three times. All paired devices will be cleared.

## What is the warranty policy?
TechGadget Pro comes with a 2-year limited warranty covering manufacturing defects.
Physical damage, water damage, and unauthorized modifications are not covered.
Contact support@techgadgetpro.com to initiate a warranty claim.

## How do I update the firmware?
Open the TechGadget app, navigate to Settings > Device > Firmware Update.
Updates download automatically over Wi-Fi and install when the device is charging.
Do not power off during an update.

## What is the return policy?
We accept returns within 30 days of purchase for any reason. The device must be in
original condition with all accessories. Refunds are processed within 5-7 business days.
"""

# Save to a local file
doc_path = Path("techgadget_faq.txt")
doc_path.write_text(sample_doc)

# --------------------------------------------------------------
print(f"✅ Sample document created: {doc_path}")
print(f"   Size: {len(sample_doc)} characters")

---

## ☁️ Part 3: Upload and Create a Vector Store

Vector store management uses the `openai` client directly, not the agents SDK.

Create the store once, then save and reuse its `vector_store.id` rather than recreating it.

Create separate stores to organize files by topic, project, or access level.

In [ ]:
print("Creating vector store...\n")

# Step 1: Create the vector store
vector_store = client.vector_stores.create(name="TechGadget FAQ")
print(f"✅ Vector store created: {vector_store.id}")

# Step 2: Upload file and poll until processing is complete
with open(doc_path, "rb") as file_handle:
    file_batch = client.vector_stores.file_batches.upload_and_poll(
        vector_store_id=vector_store.id,
        files=[file_handle]
    )

if file_batch.file_counts.failed:
    raise RuntimeError(f"File processing failed: {file_batch.file_counts.failed} file(s)")

print(f"✅ File uploaded and processed")
print(f"   Status: {file_batch.status}")
print(f"   Files: {file_batch.file_counts.completed} completed")

### 💡 Key Takeaway

`upload_and_poll` uploads, processes, and waits in one call.

The agent can't query files that are still processing.

---

## 🔍 Part 4: Query with FileSearchTool

Connect the vector store to an agent using `FileSearchTool`.

In your own project, the pattern is the same:

- Upload your documents to a vector store

- Pass that store's ID into `FileSearchTool`

- Rewrite the agent instructions for your domain

In [ ]:
instructions = (
    "You are a customer support agent for TechGadget Pro.\n"
    "Answer questions using only information from the provided documents.\n"
    "If the answer is not in the documents, say so clearly.\n"
    "Do not offer to contact support or take any action yourself: you can only answer from the documents."
)

agent = Agent(
    name="SupportAgent",
    instructions=instructions,
    model=MODEL,
    tools=[FileSearchTool(
        vector_store_ids=[vector_store.id],
        max_num_results=3
    )]
)

# --------------------------------------------------------------
print("✅ Support agent ready")

#### Test: Warranty Question

In [ ]:
print("🔍 FILE SEARCH DEMO")
print("=" * 60)

result = await Runner.run(agent, input="What is the warranty policy?")

print(f"Q: What is the warranty policy?")
print(f"File search called: {'yes' if used_file_search(result) else 'no'}")
out = result.final_output.lower()
year_ok = any(k in out for k in ["2-year", "two-year", "2 year", "two year"])
defect_ok = "defect" in out
print(f"Grounded in the FAQ (2-year term + defect coverage): {year_ok and defect_ok}")
print(f"A: {result.final_output}")

#### Test: Reset Instructions

In [ ]:
result = await Runner.run(agent, input="How do I factory reset the device?")

print(f"Q: How do I factory reset the device?")
print(f"File search called: {'yes' if used_file_search(result) else 'no'}")
out = result.final_output.lower()
hold_ok = any(k in out for k in ["10 second", "10-second", "ten second"])
buttons_ok = "volume" in out
print(f"Grounded in the FAQ (10-second hold + volume button): {hold_ok and buttons_ok}")
print(f"A: {result.final_output}")

#### Test: Question Not in Documents

In [ ]:
result = await Runner.run(agent, input="What color options are available?")

print(f"Q: What color options are available?")
print(f"File search called: {'yes' if used_file_search(result) else 'no'}")
print(f"A: {result.final_output}")
print("=" * 60)

### 💡 Key Takeaway

The tool retrieves relevant chunks.

Instructions tell the agent to answer only from those chunks.

Retrieval alone does not enforce that rule.

⚠️ **Security note:** Retrieved chunks are untrusted input.

Treat them as data, not instructions (more in Lesson 21).

---

## 🧹 Demo Cleanup

In [ ]:
# Clean up local sample files before exercises
for file in ["techgadget_faq.txt"]:
    path = Path(file)
    if path.exists():
        path.unlink()
        print(f"✅ Local file deleted: {file}")

---

## 💪 Practice Exercises

### Exercise 1: Add a Second Document

*Covers: vector stores, extending a knowledge base*

Add a second document to the knowledge base and verify the agent can answer from both.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Add a Second Document
# --------------------------------------------------------------
# Objective: Add a shipping policy document to the existing vector store.

shipping_doc = """# TechGadget Pro: Shipping Policy

## Standard Shipping
Standard shipping takes 5-7 business days and costs $4.99.
Free standard shipping on orders over $50.

## Express Shipping
Express shipping takes 2 business days and costs $12.99.

## International Shipping
We ship to 45 countries. International orders take 10-14 business days.
Customs fees are the responsibility of the customer.
"""

# TODO 1: Save shipping_doc to a file called shipping_policy.txt

# TODO 2: Upload it to the existing vector_store using
#          client.vector_stores.file_batches.upload_and_poll()
#          Hint: use vector_store_id=vector_store.id

# TODO 3: Run the agent with: "How much does express shipping cost?"
#          and print the result. Confirm used_file_search(result) is True.

# TODO 4: Run a second query that requires the FAQ document (not the shipping doc)
#          to confirm both documents are accessible from the same vector store

# TODO 5: Delete the local shipping_policy.txt when done
#          (its uploaded File is removed by the notebook's final cleanup cell)

# --- Write your code below this line ---

### Exercise 2: Build a New Knowledge Base

*Covers: vector stores, building and querying from scratch*

Create a new vector store with a different document and query it with a new agent.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Build a New Knowledge Base
# --------------------------------------------------------------
# Objective: Create a separate vector store and query it independently.

return_policy_doc = """# TechGadget Pro: Return Policy

## Standard Returns
Returns accepted within 30 days of purchase.
Items must be in original condition with all accessories included.

## Damaged Items
If your item arrived damaged, contact support within 7 days with photos.
We will send a replacement at no additional cost.

## Refund Timeline
Refunds are processed within 5-7 business days after we receive the item.
"""

# TODO 1: Save return_policy_doc to return_policy.txt

# TODO 2: Create a new vector store (give it a different name)
#          Upload return_policy.txt to it

# TODO 3: Create a new agent with the new vector store
#          Use different instructions: e.g. a returns specialist

# TODO 4: Run the agent with: "How soon should I contact support if my item arrives damaged?"
#          Confirm it answers from the return policy, not the FAQ, and that
#          used_file_search(result) is True

# TODO 5: Clean up: delete the uploaded File(s), the new vector store, and the
#          local return_policy.txt when done

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Vector stores enable RAG:**

- OpenAI chunks and indexes your files for semantic and keyword search

- Your code uploads files and queries the store

- File Search retrieves selected chunks instead of injecting the entire document

- The agent still decides when to call the tool
<br>
<br>

**Set up the store and the search tool separately:**

- Create and populate the vector store with the `openai` client

- Attach it with `FileSearchTool(vector_store_ids=[...])`

- Inspect run items when you need to prove the tool ran
<br>
<br>

**Files and vector stores are separate resources:**

- Both persist until deleted

- Delete uploaded files with `client.files.delete(...)`

- Delete the store with `client.vector_stores.delete(...)`

- In a real app, keep the store and reuse its ID until the documents change

---

## 📍 Next Step

**Notebook 10: Code Interpreter**  

Let your agent write and run Python in a sandbox to analyze data and generate outputs.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-09-file-search)

---

#### Cleanup: Delete Uploaded Files and Vector Store

Uploaded files and vector stores persist on OpenAI's servers until deleted.

Deleting the store does not delete the files you uploaded, so remove both.

Run this cell when you're done to avoid ongoing storage charges.

In a real app, keep the store and reuse its ID until the documents change.

In [ ]:
# Vector stores and uploaded Files are separate objects: delete both
uploaded_file_ids = [
    store_file.id
    for store_file in client.vector_stores.files.list(
        vector_store_id=vector_store.id
    ).data
]

for file_id in uploaded_file_ids:
    client.files.delete(file_id)

client.vector_stores.delete(vector_store.id)

print(f"✅ Uploaded files deleted: {len(uploaded_file_ids)}")
print(f"✅ Vector store deleted: {vector_store.id}")

---